# 05 — Feature Extraction

**Stage 5.** Extract physiological features from windowed WESAD signals.

- **General window statistics** (mean, std, min, max) for all signals.
- **ECG**: BPM, IBI, SDNN, SDSD, RMSSD, PNN20, PNN50 (via `heartpy`).
- **ACC**: Net acceleration, absolute integrals, peak frequencies.
- **EDA**: Slope, dynamic range.
- **EMG**: Dynamic range, absolute integral.
- **Resp**: Respiration rate, inhalation/exhalation durations, I/E ratio (via `biosppy`).
- **TEMP**: Dynamic range, slope.
- **BVP**: Peak frequency.

Input: `data/preprocessed/WESAD/S{id}_clean.pkl`  
Output: `data/features/S{id}_features.csv`

In [6]:
import sys, gc, time
from pathlib import Path
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Scripts' / 'Preprocess').exists()) / 'Scripts' / 'Preprocess'
sys.path.insert(0, str(HERE))

import numpy as np
import pandas as pd
import wesad_preprocess as wp

# Window parameters (in seconds)
WINDOW_SEC_GENERAL = 60.0
WINDOW_SEC_ACC = 5.0
SHIFT_SEC = 0.25

SUBJECTS = wp.SUBJECT_IDS

## 1. Windowing & Feature Extraction Functions

In [7]:
import matplotlib
matplotlib.use('Agg')
import scipy.signal as scisig
import scipy.stats
import heartpy as hp
import warnings
warnings.simplefilter("ignore")

# Stub out biosppy's interactive plotting (requires tkinter, which is not installed)
import sys
from unittest.mock import MagicMock
sys.modules['biosppy.inter_plotting'] = MagicMock()
sys.modules['biosppy.inter_plotting.ecg'] = MagicMock()
sys.modules['biosppy.inter_plotting.acc'] = MagicMock()
import biosppy

def extract_sliding_windows(arr, label_arr, fs, window_sec, shift_sec):
    arr = np.asarray(arr)
    label_arr = np.asarray(label_arr)
    win_len = int(round(window_sec * fs))
    shift_len = int(round(shift_sec * fs))
    if len(arr) < win_len:
        return np.array([]), np.array([])
    n_windows = (len(arr) - win_len) // shift_len + 1
    shape = (n_windows, win_len)
    strides = (arr.strides[0] * shift_len, arr.strides[0])
    if arr.ndim == 1:
        windows = np.lib.stride_tricks.as_strided(arr, shape=shape, strides=strides)
    else:
        n_channels = arr.shape[1]
        shape_multi = (n_windows, win_len, n_channels)
        strides_multi = (arr.strides[0] * shift_len, arr.strides[0], arr.strides[1])
        windows = np.lib.stride_tricks.as_strided(arr, shape=shape_multi, strides=strides_multi)
    label_shape = (n_windows, win_len)
    label_strides = (label_arr.strides[0] * shift_len, label_arr.strides[0])
    win_labels = np.lib.stride_tricks.as_strided(label_arr, shape=label_shape, strides=label_strides)
    labels = (win_labels.mean(axis=1) >= 0.5).astype(int)
    return windows, labels

def get_window_stats(data, label=-1):
    return {'mean': np.mean(data), 'std': np.std(data), 'min': np.amin(data), 'max': np.amax(data), 'label': label}

def get_net_accel(data, part):
    if part == 'wrist':
        return np.sqrt(data['ACC_x']**2 + data['ACC_y']**2 + data['ACC_z']**2)
    else:
        return np.sqrt(data['ACC_x_C']**2 + data['ACC_y_C']**2 + data['ACC_z_C']**2)

def get_slope(series):
    return scipy.stats.linregress(np.arange(len(series)), series)[0]

def get_absolute_integral(x):
    return np.sum(np.abs(x))

def get_dynamic_range(x):
    return np.max(x) / np.min(x)

def get_peak_freq(x):
    f, Pxx = scisig.periodogram(x, fs=8)
    psd_dict = {amp: freq for amp, freq in zip(Pxx, f)}
    return psd_dict[max(psd_dict.keys())]

def get_window_stats_ecg(data, label=-1):
    wd, m = hp.process(data['ECG'].dropna().reset_index(drop=True), 700)
    return {'bpm': m['bpm'], 'ibi': m['ibi'], 'sdnn': m['sdnn'], 'sdsd': m['sdsd'],
            'rmssd': m['rmssd'], 'pnn20': m['pnn20'], 'pnn50': m['pnn50']}

def get_resp_features(resp_data):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resp_rate, filtered, zeros, resp_rate_ts, resp_rate = biosppy.signals.resp.resp(resp_data, sampling_rate=700, show=False)
        extremas, values = biosppy.signals.tools.find_extrema(signal=filtered, mode='both')
    inhal_durations = []
    exhal_durations = []
    last_index = 0
    for i in range(len(extremas)):
        if values[i] * values[last_index] < 0:
            if values[last_index] < 0:
                inhal_durations.append((extremas[i] - extremas[last_index]) / 700)
            else:
                exhal_durations.append((extremas[i] - extremas[last_index]) / 700)
            last_index = i
    inhal_mean = np.mean(inhal_durations) if len(inhal_durations) > 0 else 0
    inhal_std = np.std(inhal_durations) if len(inhal_durations) > 0 else 0
    exhal_mean = np.mean(exhal_durations) if len(exhal_durations) > 0 else 0
    exhal_std = np.std(exhal_durations) if len(exhal_durations) > 0 else 0
    sum_exhal = np.sum(exhal_durations)
    ie_ratio = np.sum(inhal_durations) / sum_exhal if sum_exhal > 0 else 0
    return np.mean(resp_rate), inhal_mean, inhal_std, exhal_mean, exhal_std, ie_ratio

## 2. Feature Extraction per Window

In [8]:
def extract_features_for_subject(clean_data, shift_sec=SHIFT_SEC):
    """
    Extract features for all windows of a subject.
    Returns a DataFrame with one row per window.
    """
    samples = []
    
    # Process chest and wrist signals
    for loc in ('chest', 'wrist'):
        for name, arr in clean_data['signal'][loc].items():
            fs = clean_data['fs'][loc][name]
            lab = clean_data['label'][loc][name]
            win_sec = WINDOW_SEC_ACC if 'ACC' in name else WINDOW_SEC_GENERAL
            wins, win_labs = extract_sliding_windows(arr, lab, fs, win_sec, shift_sec)
            
            for i in range(len(wins)):
                w = wins[i]
                label = win_labs[i]
                
                # Basic stats
                wstats = get_window_stats(w, label)
                
                # ECG features
                if name == 'ECG':
                    try:
                        wstats_ecg = get_window_stats_ecg(pd.DataFrame({'ECG': w}), label)
                        for key in wstats_ecg:
                            wstats[f'ECG_{key}'] = wstats_ecg[key]
                    except:
                        pass
                
                # ACC features
                if 'ACC' in name:
                    if w.ndim == 2 and w.shape[1] == 3:
                        net_acc = np.sqrt(w[:, 0]**2 + w[:, 1]**2 + w[:, 2]**2)
                        wstats['net_acc_abs_integral'] = get_absolute_integral(net_acc)
                        wstats['ACC_x_abs_integral'] = get_absolute_integral(w[:, 0])
                        wstats['ACC_y_abs_integral'] = get_absolute_integral(w[:, 1])
                        wstats['ACC_z_abs_integral'] = get_absolute_integral(w[:, 2])
                        wstats['ACC_x_peak_freq'] = get_peak_freq(w[:, 0])
                        wstats['ACC_y_peak_freq'] = get_peak_freq(w[:, 1])
                        wstats['ACC_z_peak_freq'] = get_peak_freq(w[:, 2])
                
                # EDA features
                if 'EDA' in name:
                    wstats['EDA_slope'] = get_slope(w)
                    wstats['EDA_drange'] = get_dynamic_range(w)
                
                # EMG features
                if 'EMG' in name:
                    wstats['EMG_drange'] = get_dynamic_range(w)
                    wstats['EMG_abs_integral'] = get_absolute_integral(w)
                
                # Resp features
                if 'Resp' in name:
                    try:
                        resp_feats = get_resp_features(w)
                        wstats['Resp_C_rate'] = resp_feats[0]
                        wstats['Resp_C_Inhal_mean'] = resp_feats[1]
                        wstats['Resp_C_Inhal_std'] = resp_feats[2]
                        wstats['Resp_C_Exhal_mean'] = resp_feats[3]
                        wstats['Resp_C_Exhal_std'] = resp_feats[4]
                        wstats['Resp_C_I/E'] = resp_feats[5]
                    except:
                        pass
                
                # TEMP features
                if 'Temp' in name or 'TEMP' in name:
                    wstats['TEMP_drange'] = get_dynamic_range(w)
                    wstats['TEMP_slope'] = get_slope(w)
                
                # BVP features
                if 'BVP' in name:
                    wstats['BVP_peak_freq'] = get_peak_freq(w)
                
                wstats['subject'] = clean_data['subject']
                wstats['location'] = loc
                wstats['signal'] = name
                samples.append(wstats)
    
    return pd.DataFrame(samples)

## 3. Single-Subject Walkthrough

In [5]:
clean = wp.load_clean_subject(SUBJECTS[0])
features_df = extract_features_for_subject(clean)
print(f"Extracted {len(features_df)} feature rows for subject {clean['subject']}")
features_df.head()

KeyboardInterrupt: 

## 4. Process & Save All Subjects

In [9]:
FEATURE_OUTPUT_DIR = Path('data/features')
FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SKIP_EXTRACTED_SUBJECT = True   # True: skip subjects whose S{id}_features.csv already exists

summary = []
for sid in SUBJECTS:
    out_path = FEATURE_OUTPUT_DIR / f"S{sid}_features.csv"

    # Skip already-extracted subjects (e.g. resume after an interrupted run)
    if SKIP_EXTRACTED_SUBJECT and out_path.exists():
        print(f"S{sid}: already extracted -> skip")
        continue

    t0 = time.time()
    clean = wp.load_clean_subject(sid)
    features_df = extract_features_for_subject(clean)

    features_df.to_csv(out_path, index=False)

    summary.append({
        'subject': sid,
        'n_rows': len(features_df),
        'n_features': len(features_df.columns),
        'saved_to': out_path.name,
        'seconds': round(time.time() - t0, 2)
    })
    print(summary[-1])
    del clean, features_df; gc.collect()

pd.DataFrame(summary)

S2: already extracted -> skip
S3: already extracted -> skip
S4: already extracted -> skip
S5: already extracted -> skip
{'subject': 6, 'n_rows': 120406, 'n_features': 35, 'saved_to': 'S6_features.csv', 'seconds': 757.99}
{'subject': 7, 'n_rows': 120182, 'n_features': 35, 'saved_to': 'S7_features.csv', 'seconds': 635.85}
{'subject': 8, 'n_rows': 120788, 'n_features': 35, 'saved_to': 'S8_features.csv', 'seconds': 673.03}
{'subject': 9, 'n_rows': 120126, 'n_features': 35, 'saved_to': 'S9_features.csv', 'seconds': 665.38}
{'subject': 10, 'n_rows': 124606, 'n_features': 35, 'saved_to': 'S10_features.csv', 'seconds': 823.48}
{'subject': 11, 'n_rows': 121862, 'n_features': 35, 'saved_to': 'S11_features.csv', 'seconds': 830.18}
{'subject': 13, 'n_rows': 121750, 'n_features': 35, 'saved_to': 'S13_features.csv', 'seconds': 1482.35}
{'subject': 14, 'n_rows': 121806, 'n_features': 35, 'saved_to': 'S14_features.csv', 'seconds': 1430.42}
{'subject': 15, 'n_rows': 122142, 'n_features': 35, 'saved_to'

,subject,n_rows,n_features,saved_to,seconds
0,6,120406,35,S6_features.csv,757.99
1,7,120182,35,S7_features.csv,635.85
2,8,120788,35,S8_features.csv,673.03
3,9,120126,35,S9_features.csv,665.38
4,10,124606,35,S10_features.csv,823.48
5,11,121862,35,S11_features.csv,830.18
6,13,121750,35,S13_features.csv,1482.35
7,14,121806,35,S14_features.csv,1430.42
8,15,122142,35,S15_features.csv,1495.86
9,16,121470,35,S16_features.csv,1133.43


## 5. Inspect Features by Signal Type

In [ ]:
# Inspect non-null extracted features grouped by signal type
for sig in features_df['signal'].unique():
    sample = features_df[features_df['signal'] == sig].dropna(how='all', axis=1)
    print(f"=== Signal: {sig} (Rows: {len(sample)}) ===")
    display(sample.head(2))